# C4 adnotare corpus exercițiu

Notebook placeholder pentru exercițiul individual din C4.

In [63]:
import json

comments = []
with open("../../data/cleaned/student_01_youtube_clean.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        comments.append(json.loads(line))

comments[5:10]

[{'id': 'yt_TFSXVaF5HnM_Ugyz3WUiMRVUbN1R60N4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'RecorderRomania',
  'text_raw': 'Amuzant, mina este PA dar salariul celor din conducere merge inainte',
  'video_id': 'TFSXVaF5HnM',
  'video_title': 'Cazul salina Praid. Dezastrul pentru care nu a plătit nimeni #shorts',
  'video_date': '2026-05-05',
  'comment_date': '2026-05-05',
  'likes': 8,
  'collected_at': '2026-05-10',
  'text': 'Amuzant, mina este PA dar salariul celor din conducere merge inainte',
  'lang': 'ro'},
 {'id': 'yt_TFSXVaF5HnM_UgxydAUp9cebAT8qWJN4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'RecorderRomania',
  'text_raw': 'Auziți cineva sa ii creadă pe alde Bolojan și Nicușor Dan că lupta împotriva sistemului ...pai ei sunt primi care nu mai vor să audă de așa ceva ...  Iar acolo fără să fiu tendențios erwlau ungurii la  conducere ...de.asra se.s8 astupa ...',
  'video_id': 'TFSXVaF5HnM',
  'video_title': 'Cazul salina Praid. Dezastrul pentru 

In [64]:
from openai import OpenAI
import google.generativeai as genai
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Create Groq client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq client initialized ✅")


Groq client initialized ✅


AXIS 1: Sentiment
- 0 = Negative
- 1 = Neutral or mixed
- 2 = Positive

AXIS 2: Political orientation
- 0 = Left-leaning
- 1 = Center-right
- 2 = Right-leaning

In [75]:
SYSTEM = """You are an annotation assistant for social science research."""

PROMPT = """
Analyze the following YouTube comment and assign values on three axes.


AXIS 1: Political orientation
- 0 = Left-leaning
- 1 = Center-right
- 2 = Right-leaning

AXIS 2: Target of evaluation
- 0 = Individuals
- 1 = Institutions / systems
- 2 = Society / collective

Rules:
- Choose exactly ONE value for each axis.
- Return ONLY valid JSON.
- Do NOT explain.
- Do NOT ask questions.


FORMAT:
{{
  "Political orientation": 0,
  "Target of evaluation": 0
}}


COMMENT:
{comment}
"""



In [76]:

def ask(provider, model, prompt, system=""):
    if provider == "groq":
        client = OpenAI(
            api_key=os.getenv("GROQ_API_KEY"),
            base_url="https://api.groq.com/openai/v1"
        )

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ]
        )

        return response.choices[0].message.content


In [77]:
import json

for i, c in enumerate(comments[:5], 1):
    print("=" * 80)
    print(f"Comentariu {i} – TEXT ORIGINAL:")
    print(c["text"])
    print("-" * 80)
    print("ANALIZĂ:")

    raw_output = ask(
        provider=provider,
        model=model,
        system=SYSTEM,
        prompt=PROMPT.format(comment=c["text"])
    )

    try:
        data = json.loads(raw_output)

        sentiment = int(data["Sentiment"])
        orientation = int(data["Political orientation"])
        target = int(data["Target of evaluation"])

        print(f"- Sentiment: {sentiment} ({SENTIMENT_MAP[sentiment]})")
        print(f"- Orientare politică: {orientation} ({POLITICAL_ORIENTATION_MAP[orientation]})")
        print(f"- Ținta evaluării: {target} ({TARGET_MAP[target]})")

    except Exception:
        print("Eroare la parsare JSON. Output brut:")
        print(raw_output)

    print("\n")


Comentariu 1 – TEXT ORIGINAL:
Sfat util pentru dobitocii care au votat 36 de ani cu PSD(fostul PCR) acum au opțiunea să voteze cu AUR(PSD2).
--------------------------------------------------------------------------------
ANALIZĂ:
Eroare la parsare JSON. Output brut:
{
  "Political orientation": 0,
  "Target of evaluation": 0
}


Comentariu 2 – TEXT ORIGINAL:
Discursul lui Farfuridi, de I.Luca Caragiale "Din două una, dați-mi voie: ori să se revizuiască, primesc! Dar să nu se schimbe nimic; ori să nu se revizuiască, primesc! Dar atunci să se schimbe pe aici pe colo, și anume în punctele… esenţiale."
--------------------------------------------------------------------------------
ANALIZĂ:
Eroare la parsare JSON. Output brut:
{
  "Political orientation": 0,
  "Target of evaluation": 0
}


Comentariu 3 – TEXT ORIGINAL:
Nenorociților , distrugeți o țară !!! Doamne , pedepsește i pe trădători !!
--------------------------------------------------------------------------------
ANALIZĂ:
Eroare

## Tabel sinteză 


| Axă | 0 | 1 | 2 |
|-----|---|---|---|
| **Political orientation** | Stânga | Centru‑dreapta | Dreapta |
| **Target of evaluation** | Indivizi | Instituții / sisteme | Societate / colectiv |

---

### Comentarii "adnotate"

| # | Comentariu (scurt) | Political orientation | Target of evaluation | Tip discursiv |
|---|-------------------|----------------------|---------------------|---------------|
| 1 | „Sfat util pentru dobitocii…” | 0 (Stânga) | 0 (Indivizi) | Critică politică personalizată |
| 2 | Discurs Caragiale | 0 (Stânga) | 0 (Indivizi) | Ironie culturală cu țintă individuală |
| 3 | „Nenorociților, distrugeți o țară!” | 2 (Dreapta) | 1 (Instituții) | Critică sistemică vehementă |
| 4 | „Distrus tot ce mai poate produce țara” | 2 (Dreapta) | 2 (Societate) | Discurs moralizator colectiv |
| 5 | „Își iau salarii de la stat” | 2 (Dreapta) | 0 (Indivizi) | Atac personalizat anti‑elite |


It is quite difficult to interpret, given the quantity of data, and the request I submitted to groq...
Groq clearly does not know the difference between right and left : ))
I can say that from the table we see predominantly individual blaming and targeting, a grup of individuals as we can see. Only one system blaming.
It does not recognise symbolic speach, references and subtile nuances. 
Not a great model and I am happy about it! Low performance, means good for society.  